# SigilAGISilver — ARC-AGI-3 final submission notebook

This notebook writes exactly one submit-ready artifact: `/kaggle/working/my_agent.py`.

Key fixes:

- no `submission.parquet` / `submission.csv` / `submission.zip` pollution
- no notebook import of `my_agent.py`, so no `__pycache__` selection bug
- final sanitizer deletes `__pycache__` and `*.pyc`
- output contract is the ARC agent file only


In [ ]:
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
AGENT = WORK / 'my_agent.py'
AGENT.write_text('# ============================================================\n# SigilAGISilver / FORGE v29 — ARC-AGI-3 REAL SUBMISSION AGENT\n# Output contract: this file is the submitted artifact: my_agent.py\n# Runtime contract: no internet, no auxiliary generated submission file.\n# Strategy: deterministic adaptive exploration, state-diff targeting,\n#           safe action selection, cache-free submission hygiene.\n# ============================================================\n\nfrom __future__ import annotations\n\nimport hashlib\nimport math\nimport random\nimport time\nfrom collections import Counter, deque\nfrom typing import Any, Iterable, Optional\n\ntry:\n    import numpy as np\nexcept Exception:  # pragma: no cover - competition image normally has numpy\n    np = None\n\n# ARC runtime imports. The same file must tolerate being loaded either from the\n# repo root or from inside an agents/ package. Do not import this file in the\n# notebook preflight; ast.parse is used instead to avoid __pycache__ artifacts.\ntry:\n    from arcengine import FrameData, GameAction, GameState\nexcept Exception:  # pragma: no cover\n    try:\n        from agents.structs import FrameData, GameAction, GameState  # type: ignore\n    except Exception:  # offline syntax fallback only\n        FrameData = Any  # type: ignore\n        class _FallbackState:\n            NOT_PLAYED = "NOT_PLAYED"\n            NOT_FINISHED = "NOT_FINISHED"\n            WIN = "WIN"\n            GAME_OVER = "GAME_OVER"\n        GameState = _FallbackState()  # type: ignore\n        class _FallbackAction:\n            RESET = None\n        GameAction = _FallbackAction  # type: ignore\n\ntry:\n    from agents.agent import Agent\nexcept Exception:  # pragma: no cover\n    try:\n        from .agent import Agent  # type: ignore\n    except Exception:\n        try:\n            from ..agent import Agent  # type: ignore\n        except Exception:  # offline syntax fallback only\n            class Agent:  # type: ignore\n                MAX_ACTIONS = 80\n                def __init__(self, *args: Any, **kwargs: Any) -> None:\n                    self.game_id = kwargs.get("game_id", "offline")\n                    self.action_counter = 0\n                @property\n                def name(self) -> str:\n                    return self.__class__.__name__.lower()\n\n\ndef _state_name(state: Any) -> str:\n    return str(getattr(state, "name", getattr(state, "value", state)))\n\n\ndef _safe_int(x: Any, default: int = 0) -> int:\n    try:\n        return int(x)\n    except Exception:\n        return default\n\n\ndef _hash_frame(frame: Any) -> str:\n    try:\n        if np is not None:\n            arr = np.asarray(frame, dtype=np.int64)\n            return hashlib.blake2b(arr.tobytes(), digest_size=8).hexdigest()\n        return hashlib.blake2b(repr(frame).encode("utf-8", "replace"), digest_size=8).hexdigest()\n    except Exception:\n        return hashlib.blake2b(str(time.time()).encode(), digest_size=8).hexdigest()\n\n\nclass MyAgent(Agent):\n    """Deterministic adaptive ARC-AGI-3 agent.\n\n    The policy is deliberately model-free and submission-safe:\n    1. reset exactly when required;\n    2. mine frame differences and non-background components;\n    3. probe all available simple actions before complex clicks;\n    4. repeat actions that increase score/levels;\n    5. never emit invalid complex coordinates;\n    6. keep reasoning compact and JSON-serializable.\n    """\n\n    MAX_ACTIONS = 80\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        gid = str(getattr(self, "game_id", kwargs.get("game_id", "unknown")))\n        seed = int(hashlib.blake2b(gid.encode("utf-8", "replace"), digest_size=4).hexdigest(), 16)\n        self.rng = random.Random(seed)\n        self.best_score = -1\n        self.last_score = -1\n        self.last_hash = ""\n        self.no_change_streak = 0\n        self.win_seen = False\n        self.success_replay: deque[tuple[str, tuple[int, int] | None]] = deque(maxlen=8)\n        self.click_targets: list[tuple[int, int]] = []\n        self.simple_plan = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"]\n        self.complex_name = "ACTION6"\n        self._plan_cursor = 0\n\n    @property\n    def name(self) -> str:\n        base = getattr(super(), "name", self.__class__.__name__.lower())\n        return f"{base}.forge_v29_submission_fixed"\n\n    # -----------------------------\n    # Runtime observation utilities\n    # -----------------------------\n    def _score(self, frame: Any) -> int:\n        return max(\n            _safe_int(getattr(frame, "levels_completed", 0), 0),\n            _safe_int(getattr(frame, "score", 0), 0),\n        )\n\n    def _frame_matrix(self, frame_payload: Any) -> Any:\n        if frame_payload is None:\n            return None\n        if np is None:\n            return frame_payload\n        try:\n            arr = np.asarray(frame_payload)\n            if arr.size == 0:\n                return None\n            # Convert CHW -> HWC if the first axis is channel-like.\n            if arr.ndim == 3 and arr.shape[0] <= 8 and arr.shape[1] > 8 and arr.shape[2] > 8:\n                arr = np.moveaxis(arr, 0, -1)\n            return arr\n        except Exception:\n            return None\n\n    def _gray(self, arr: Any) -> Any:\n        if arr is None or np is None:\n            return None\n        try:\n            if arr.ndim == 3:\n                return arr.astype("int64").sum(axis=-1)\n            if arr.ndim == 2:\n                return arr.astype("int64")\n            return None\n        except Exception:\n            return None\n\n    def _scale_xy(self, x: float, y: float, width: int, height: int) -> tuple[int, int]:\n        width = max(1, width)\n        height = max(1, height)\n        sx = int(round(max(0.0, min(width - 1.0, x)) * 63.0 / max(1, width - 1)))\n        sy = int(round(max(0.0, min(height - 1.0, y)) * 63.0 / max(1, height - 1)))\n        return max(0, min(63, sx)), max(0, min(63, sy))\n\n    def _interesting_targets(self, frames: list[Any], latest_frame: Any) -> list[tuple[int, int]]:\n        arr = self._frame_matrix(getattr(latest_frame, "frame", None))\n        gray = self._gray(arr)\n        if gray is None or np is None:\n            # Stable fallback grid. These are safe complex-action coordinates.\n            fallback = [\n                (32, 32), (16, 16), (48, 16), (16, 48), (48, 48),\n                (32, 16), (32, 48), (16, 32), (48, 32),\n                (8, 8), (56, 8), (8, 56), (56, 56),\n            ]\n            return fallback\n\n        h, w = gray.shape[:2]\n        points: list[tuple[int, int]] = []\n\n        # Background estimate: border mode, falling back to global mode.\n        try:\n            border = np.concatenate([gray[0, :], gray[-1, :], gray[:, 0], gray[:, -1]])\n            vals, counts = np.unique(border, return_counts=True)\n            bg = vals[int(np.argmax(counts))]\n        except Exception:\n            vals, counts = np.unique(gray, return_counts=True)\n            bg = vals[int(np.argmax(counts))]\n\n        mask = gray != bg\n        if int(mask.sum()) > 0:\n            ys, xs = np.where(mask)\n            # Centroid and bounding box anchors.\n            points.append(self._scale_xy(float(xs.mean()), float(ys.mean()), w, h))\n            x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())\n            for px, py in [\n                ((x0 + x1) / 2, (y0 + y1) / 2),\n                (x0, y0), (x1, y0), (x0, y1), (x1, y1),\n                ((x0 + x1) / 2, y0), ((x0 + x1) / 2, y1),\n                (x0, (y0 + y1) / 2), (x1, (y0 + y1) / 2),\n            ]:\n                points.append(self._scale_xy(px, py, w, h))\n\n            # Component-like coarse clusters by quantized bins.\n            bins: dict[tuple[int, int], list[tuple[int, int]]] = {}\n            qx = max(1, w // 8)\n            qy = max(1, h // 8)\n            for x, y in zip(xs[:: max(1, len(xs) // 512)], ys[:: max(1, len(ys) // 512)]):\n                key = (int(x) // qx, int(y) // qy)\n                bins.setdefault(key, []).append((int(x), int(y)))\n            for group in sorted(bins.values(), key=len, reverse=True)[:8]:\n                gx = sum(p[0] for p in group) / len(group)\n                gy = sum(p[1] for p in group) / len(group)\n                points.append(self._scale_xy(gx, gy, w, h))\n\n        # Change centroid from previous rendered frame.\n        try:\n            if len(frames) >= 2:\n                prev = self._gray(self._frame_matrix(getattr(frames[-2], "frame", None)))\n                if prev is not None and prev.shape == gray.shape:\n                    diff = np.abs(gray.astype("int64") - prev.astype("int64"))\n                    dmask = diff > 0\n                    if int(dmask.sum()) > 0:\n                        dys, dxs = np.where(dmask)\n                        points.insert(0, self._scale_xy(float(dxs.mean()), float(dys.mean()), w, h))\n        except Exception:\n            pass\n\n        # Scan lattice gives coverage for hidden click targets.\n        for gy in [8, 16, 24, 32, 40, 48, 56]:\n            for gx in [8, 16, 24, 32, 40, 48, 56]:\n                if (gx, gy) not in points:\n                    points.append((gx, gy))\n\n        # De-duplicate while preserving order.\n        seen: set[tuple[int, int]] = set()\n        out: list[tuple[int, int]] = []\n        for p in points:\n            x, y = max(0, min(63, int(p[0]))), max(0, min(63, int(p[1])))\n            if (x, y) not in seen:\n                seen.add((x, y))\n                out.append((x, y))\n        return out[:64]\n\n    # -----------------------------\n    # Action utilities\n    # -----------------------------\n    def _all_actions(self) -> list[Any]:\n        try:\n            return list(GameAction)\n        except Exception:\n            return []\n\n    def _action_name(self, action: Any) -> str:\n        return str(getattr(action, "name", action)).upper()\n\n    def _available_actions(self, latest_frame: Any) -> list[Any]:\n        avail = getattr(latest_frame, "available_actions", None)\n        if avail:\n            out = list(avail)\n        else:\n            out = self._all_actions()\n        # Remove duplicates by name.\n        dedup: dict[str, Any] = {}\n        for a in out:\n            dedup[self._action_name(a)] = a\n        return list(dedup.values())\n\n    def _find_action(self, latest_frame: Any, preferred_name: str, allow_reset: bool = False) -> Optional[Any]:\n        preferred_name = preferred_name.upper()\n        for a in self._available_actions(latest_frame):\n            name = self._action_name(a)\n            if name == preferred_name and (allow_reset or name != "RESET"):\n                return a\n        # Fallback to global enum lookup if available_actions is missing/incomplete.\n        for a in self._all_actions():\n            name = self._action_name(a)\n            if name == preferred_name and (allow_reset or name != "RESET"):\n                return a\n        return None\n\n    def _make_action(self, action: Any, xy: Optional[tuple[int, int]], reason: str) -> Any:\n        if action is None:\n            return action\n        try:\n            is_complex = bool(action.is_complex())\n        except Exception:\n            is_complex = self._action_name(action) == self.complex_name\n\n        if is_complex:\n            x, y = xy if xy is not None else (32, 32)\n            payload = {"x": max(0, min(63, int(x))), "y": max(0, min(63, int(y)))}\n            try:\n                action.set_data(payload)\n            except Exception:\n                pass\n            try:\n                action.reasoning = {"p": "forge_v29", "xy": payload, "why": reason[:96]}\n            except Exception:\n                pass\n        else:\n            try:\n                action.reasoning = f"forge_v29:{reason[:96]}"\n            except Exception:\n                pass\n        return action\n\n    def _reset_action(self, latest_frame: Any, reason: str = "required_reset") -> Any:\n        action = self._find_action(latest_frame, "RESET", allow_reset=True)\n        if action is None:\n            # Last resort for runtimes where RESET is an enum attribute.\n            action = getattr(GameAction, "RESET", None)\n        return self._make_action(action, None, reason)\n\n    def _best_simple_probe(self, latest_frame: Any) -> Optional[Any]:\n        available_names = {self._action_name(a) for a in self._available_actions(latest_frame)}\n        candidates = [n for n in self.simple_plan if n in available_names]\n        if not candidates:\n            candidates = self.simple_plan[:]\n        # Movement-first schedule with occasional action/use buttons.\n        idx = self._plan_cursor % max(1, len(candidates))\n        self._plan_cursor += 1\n        return self._find_action(latest_frame, candidates[idx])\n\n    # -----------------------------\n    # Required interface\n    # -----------------------------\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        state = _state_name(getattr(latest_frame, "state", ""))\n        if state == "WIN":\n            self.win_seen = True\n            return True\n        return _safe_int(getattr(self, "action_counter", 0), 0) >= self.MAX_ACTIONS\n\n    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:\n        state = _state_name(getattr(latest_frame, "state", ""))\n        counter = _safe_int(getattr(self, "action_counter", 0), 0)\n\n        # Required starts/restarts only.\n        if state in {"NOT_PLAYED", "GAME_OVER"}:\n            return self._reset_action(latest_frame, f"state:{state}")\n\n        # Update memory from latest observation.\n        score = self._score(latest_frame)\n        fhash = _hash_frame(getattr(latest_frame, "frame", None))\n        changed = fhash != self.last_hash\n        if self.last_hash:\n            self.no_change_streak = 0 if changed else self.no_change_streak + 1\n        self.last_hash = fhash\n\n        if score > self.best_score:\n            # A recent action improved the game; replay related action pressure.\n            self.best_score = score\n            try:\n                last_input = getattr(latest_frame, "action_input", None)\n                last_id = getattr(last_input, "id", None)\n                last_name = self._action_name(last_id)\n                if last_name and last_name != "RESET":\n                    self.success_replay.append((last_name, None))\n            except Exception:\n                pass\n\n        self.click_targets = self._interesting_targets(frames, latest_frame)\n\n        # If success was detected, try to exploit the same simple action briefly.\n        if self.success_replay and counter % 4 in {0, 1}:\n            name, xy = self.success_replay[-1]\n            action = self._find_action(latest_frame, name)\n            if action is not None:\n                return self._make_action(action, xy, f"exploit_best_score:{self.best_score}")\n\n        # Prefer simple actions for dynamics discovery; switch faster if no visual changes.\n        simple_window = 36 if self.no_change_streak < 3 else 18\n        if counter < simple_window or counter % 3 != 0:\n            action = self._best_simple_probe(latest_frame)\n            if action is not None:\n                return self._make_action(action, None, f"simple_probe:{counter}:score:{score}:chg:{changed}")\n\n        # Complex click/point action targeted at meaningful pixels and grid lattice.\n        complex_action = self._find_action(latest_frame, self.complex_name)\n        if complex_action is not None:\n            if not self.click_targets:\n                self.click_targets = [(32, 32)]\n            target = self.click_targets[(counter + self.no_change_streak) % len(self.click_targets)]\n            return self._make_action(complex_action, target, f"target_probe:{target}:score:{score}")\n\n        # Fallback: any non-reset available action.\n        for action in self._available_actions(latest_frame):\n            if self._action_name(action) != "RESET":\n                return self._make_action(action, None, "fallback_non_reset")\n\n        return self._reset_action(latest_frame, "no_available_non_reset")\n\n\n# Compatibility aliases for different Kaggle/ARC loader conventions.\nclass SigilAGISilver(MyAgent):\n    pass\n\n\nclass Forge(MyAgent):\n    pass\n\n\nclass FORGE(MyAgent):\n    pass\n\n\nAVAILABLE_AGENTS = {\n    "myagent": MyAgent,\n    "my_agent": MyAgent,\n    "sigilagisilver": SigilAGISilver,\n    "forge": Forge,\n    "FORGE": FORGE,\n}\n', encoding='utf-8')
print(f'[WRITE OK] {AGENT} ({AGENT.stat().st_size:,} bytes)')


In [ ]:

# ============================================================
# FINAL ARC-AGI-3 OUTPUT SANITIZER / PREFLIGHT
# Must remain the final executed cell.
# ============================================================
from pathlib import Path
import ast
import shutil

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
AGENT = WORK / 'my_agent.py'

assert AGENT.exists(), 'FATAL: /kaggle/working/my_agent.py does not exist'
assert AGENT.stat().st_size > 10_000, 'FATAL: my_agent.py is too small / incomplete'

src = AGENT.read_text(encoding='utf-8', errors='replace')
ast.parse(src)  # syntax validation without importing; avoids generating .pyc

required = [
    'class MyAgent(Agent)',
    'def is_done(',
    'def choose_action(',
    'GameAction',
    'GameState',
    'AVAILABLE_AGENTS',
]
missing = [x for x in required if x not in src]
assert not missing, f'FATAL: my_agent.py missing required agent signals: {missing}'

# Remove every selectable bytecode/cache artifact that caused the Kaggle UI to submit the wrong file.
for p in list(WORK.rglob('__pycache__')):
    shutil.rmtree(p, ignore_errors=True)
for p in list(WORK.rglob('*.pyc')):
    try:
        p.unlink()
    except FileNotFoundError:
        pass

# Remove wrong-competition outputs from previous notebook variants.
for bad_name in [
    'submission.csv',
    'submission.parquet',
    'submission.zip',
    'adapter_config.json',
    'logs.log',
]:
    p = WORK / bad_name
    if p.exists() and p.is_file():
        p.unlink()

print('[FINAL OUTPUT CHECK]')
for p in sorted(WORK.iterdir()):
    if p.name.startswith('.'):
        continue
    print(' -', p.name)

assert AGENT.exists(), 'FATAL: sanitizer removed my_agent.py'
print('[OK] Submit using Output -> my_agent.py -> Submit to Competition')
print('[OK] Do not use MCP. Do not select __pycache__.')
